---
title: "Part 3: Combining and Reshaping Data"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/03-data-analysis/03-combining-reshaping.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/03-data-analysis/03-combining-reshaping.ipynb)

**DS-MLOps Data Analysis**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook continues from Part 2 (`02-pandas-operations.ipynb`). Real analyses rarely live in one table: a second file with reference data shows up, a result needs splitting by category, and a wide summary table is easier to read than a long one. This part covers the four operations that handle those situations: concatenation, merging, `groupby`, and pivoting.

Alongside the student exam results, we use a small synthetic table mapping each `school_id` to a region, the second table every `merge` example needs.

Part 4 (`04-time-series.ipynb`) continues with dates and time-indexed data.

| Topic | Why it matters |
|---|---|
| **`pd.concat`** | Stack DataFrames on top of each other, or side by side |
| **`pd.merge`** | Combine two tables on a shared key, the same idea as a SQL join |
| **`groupby`** | Split a dataset into groups, apply a function to each, combine the results |
| **`pivot_table`** | Reshape a long result into a wide, readable summary table |

## Callout guide

<table style='border-collapse:collapse;width:60%'>
<tr><td style='padding:6px 12px;background:#EAF3FA;border-left:4px solid #0369A1'><i class="bi bi-info-circle-fill" style="color:#0369A1"></i> <b>Key Concept</b></td><td style='padding:6px 12px'>Core idea with precise definition</td></tr>
<tr><td style='padding:6px 12px;background:#EAF7F0;border-left:4px solid #059669'><i class="bi bi-journal-code" style="color:#059669"></i> <b>Example</b></td><td style='padding:6px 12px'>Worked illustration</td></tr>
<tr><td style='padding:6px 12px;background:#FFF1E6;border-left:4px solid #EA580C'><i class="bi bi-puzzle-fill" style="color:#EA580C"></i> <b>Activity</b></td><td style='padding:6px 12px'>Hands-on challenge to try yourself</td></tr>
<tr><td style='padding:6px 12px;background:#FEF2F2;border-left:4px solid #DC2626'><i class="bi bi-bug-fill" style="color:#DC2626"></i> <b>Common Mistake</b></td><td style='padding:6px 12px'>Bug pattern to watch for</td></tr>
<tr><td style='padding:6px 12px;background:#F4F5F6;border-left:4px solid #6B7280'><i class="bi bi-lightbulb-fill" style="color:#6B7280"></i> <b>Pro Tip</b></td><td style='padding:6px 12px'>Production-grade habit</td></tr>
</table>

## Learning Objectives

By the end of Part 3 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Stack DataFrames with `pd.concat`, by row and by column | Sec. 1 |
| 2 | Combine two tables on a shared key with `pd.merge`, and choose the right `how` | Sec. 2 |
| 3 | Split a dataset into groups and aggregate each one with `groupby` | Sec. 3 |
| 4 | Reshape a grouped result into a wide summary with `pivot_table` | Sec. 4 |

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/students_exams_results.csv")
df["average_marks"] = (df["mathematics_marks"] + df["english_marks"] + df["science_marks"]) / 3

regions = ["North", "South", "East", "West"]
school_ids = sorted(df["school_id"].unique())
schools = pd.DataFrame({"school_id": school_ids, "region": [regions[i % 4] for i in range(len(school_ids))]})
df.head()

## 1. Concatenating DataFrames with `pd.concat`

`pd.concat` stacks DataFrames together. By default it stacks rows on top of each other (`axis=0`), the operation you need when the same columns show up in two separate files or two separate batches:

In [ ]:
male_students = df[df["gender"] == "M"]
female_students = df[df["gender"] == "F"]

recombined = pd.concat([male_students, female_students], axis=0)
print(f"male: {len(male_students)}, female: {len(female_students)}, combined: {len(recombined)}")

Passing `axis=1` stacks columns side by side instead, matching rows up by index. This is the shape you get when a calculation is done separately and needs joining back onto the original table:

In [ ]:
pass_fail = (df["average_marks"] >= 0.6).rename("passed")
with_pass_fail = pd.concat([df[["student_id", "average_marks"]], pass_fail], axis=1)
with_pass_fail.head()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Concatenating by column without matching indexes first</span><br><br>
<code>pd.concat([df_a, df_b], axis=1)</code> lines rows up by index position, not by any shared key. If <code>df_a</code> and <code>df_b</code> were filtered, sorted, or reset differently beforehand, row 0 in one might not be row 0 in the other, and the result silently combines the wrong rows. <code>pd.merge</code> (Sec. 2) is the safer choice whenever there is an actual key column to join on.
</div>

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Split and Recombine</span><br><br>

<b>Goal:</b> Split <code>df</code> into two DataFrames by <code>caste</code>: one for <code>"BC"</code> and one for everything else, using <code>.isin()</code> or a boolean mask. Concatenate them back with <code>pd.concat</code> and confirm the row count matches the original.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>bc_only = df[df["caste"] == "BC"]
not_bc = df[df["caste"] != "BC"]
recombined = pd.concat([bc_only, not_bc], axis=0)
len(recombined) == len(df)</pre>
</div>

In [ ]:
# TODO: split df by caste == "BC", concat back, and confirm the row count matches
...

## 2. Joining Data with `pd.merge`

`pd.merge` combines two tables on a shared key, the same idea as a SQL join. The `schools` table built above has one row per `school_id`; merging it onto `df` attaches a `region` to every student's row:

In [ ]:
df_with_region = pd.merge(df, schools, on="school_id", how="left")
df_with_region[["student_id", "school_id", "region"]].head()

The `how` argument decides which rows survive when a key on one side has no match on the other. `schools` here has a row for every `school_id` in `df`, so every `how` gives the same result; the difference only shows up once the two tables disagree:

<div style='background:#EAF7F0;border-left:5px solid #059669;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#059669;font-weight:bold'><i class="bi bi-journal-code"></i> Example: When <code>how</code> actually changes the result</span><br><br>
Merging <code>df</code> against only half of <code>schools</code> with <code>how="inner"</code> keeps only students whose school is in that half. The same merge with <code>how="left"</code> keeps every student, with <code>region</code> set to <code>NaN</code> wherever the school was missing from the smaller table.
</div>

In [ ]:
half_schools = schools[schools["region"].isin(["North", "South"])]

inner_result = pd.merge(df, half_schools, on="school_id", how="inner")
left_result = pd.merge(df, half_schools, on="school_id", how="left")
missing = left_result["region"].isna().sum()
print(f"inner: {len(inner_result)} rows, left: {len(left_result)} rows, missing region: {missing}")

The four join types differ only in which rows they keep, never in how rows are matched:

In [ ]:
from ark.plot.diagrams import merge_join_types_diagram

merge_join_types_diagram();

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Forgetting that an unmatched key produces <code>NaN</code>, not a dropped row</span><br><br>
With <code>how="left"</code>, a student whose <code>school_id</code> has no match in <code>schools</code> still gets a row, just with <code>NaN</code> in every column that came from <code>schools</code>. Code that assumes every row has a real <code>region</code> after a left merge will silently miscount or mis-group those rows later, usually in a <code>groupby</code> a few cells down.
</div>

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Count Schools Per Region</span><br><br>

<b>Goal:</b> Merge <code>df</code> with <code>schools</code> using <code>how="left"</code>, then use <code>.value_counts()</code> on the resulting <code>region</code> column to see how many student rows fall in each region.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, schools, on="school_id", how="left")
merged["region"].value_counts()</pre>
</div>

In [ ]:
# TODO: merge df with schools (how="left"), then value_counts on region
...

## 3. Group By Operations

`groupby` splits a dataset into groups by a key, applies a function to each group independently, and combines the results back into one table, one row per group:

In [ ]:
from ark.plot.diagrams import groupby_split_apply_combine_diagram

groupby_split_apply_combine_diagram();

In [ ]:
df.groupby("caste")["average_marks"].mean()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>groupby</code> computes nothing until you aggregate</span><br><br>
<code>df.groupby("caste")</code> on its own returns a <code>DataFrameGroupBy</code> object, a plan for splitting the data, not a result. Nothing is actually computed until a method like <code>.mean()</code>, <code>.sum()</code>, or <code>.agg()</code> is called on it, the same lazy-then-compute pattern you will see again in Part 5's Polars notebook.
</div>

`.agg()` works after `groupby` exactly as it did in Part 2, computing several statistics per group in one call:

In [ ]:
df.groupby("caste")["average_marks"].agg(["mean", "std", "count"])

Grouping by more than one column splits into one group per combination, here gender within caste:

In [ ]:
df.groupby(["caste", "gender"])["average_marks"].mean()

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>observed=True</code> when grouping a <code>category</code> column</span><br><br>
Grouping a column with <code>category</code> dtype (Part 2, Sec. 2) by default includes every category the dtype knows about, even ones with zero rows in the current data, padding the result with empty groups. Passing <code>observed=True</code> to <code>groupby</code> keeps only the categories that actually appear.
</div>

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Dropout Rate by Caste</span><br><br>

<b>Goal:</b> Group <code>df</code> by <code>caste</code> and compute, for each group, what fraction of students have <code>continue_drop == "drop"</code>. <code>(series == "drop").mean()</code> gives a fraction directly, no separate count and divide needed.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>df.groupby("caste")["continue_drop"].apply(lambda s: (s == "drop").mean())</pre>
</div>

In [ ]:
# TODO: group by caste, compute the drop fraction per group
...

## 4. Pivoting Data

`pivot_table` is `groupby` plus a reshape, in one call: it groups by one column, splits further by another, and lays the second grouping out as columns instead of stacking it into more rows, much easier to scan as a summary:

In [ ]:
pd.pivot_table(df, index="caste", columns="gender", values="average_marks", aggfunc="mean")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>pivot_table</code> is <code>groupby</code> with a different output shape</span><br><br>
<code>df.groupby(["caste", "gender"])["average_marks"].mean()</code> from Sec. 3 and the <code>pivot_table</code> call above compute the exact same numbers. <code>groupby</code> returns them stacked into a long Series with a two-level index; <code>pivot_table</code> lays the second key out across columns instead. Reach for <code>pivot_table</code> specifically when the result is meant to be read by a person, not processed further by code.
</div>

`aggfunc` accepts a list too, giving more than one statistic per cell:

In [ ]:
pd.pivot_table(df, index="caste", columns="gender", values="average_marks", aggfunc=["mean", "count"])

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Region by Caste Pivot</span><br><br>

<b>Goal:</b> Merge <code>df</code> with <code>schools</code> (Sec. 2), then build a pivot table with <code>region</code> as the index, <code>caste</code> as the columns, and the mean <code>average_marks</code> in each cell.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, schools, on="school_id", how="left")
pd.pivot_table(merged, index="region", columns="caste", values="average_marks", aggfunc="mean")</pre>
</div>

In [ ]:
# TODO: merge with schools, then pivot region x caste on mean average_marks
...

## Capstone: Regional Performance Report

Combine every operation from this notebook into one report: a merge to bring in region, a groupby to summarize, and a pivot to lay the summary out for reading.

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Regional Performance Report</span><br><br>

<b>Goal:</b>
<ol>
<li>Merge <code>df</code> with <code>schools</code> on <code>school_id</code>, keeping every student (Sec. 2)</li>
<li>Group the merged table by <code>region</code> and compute the mean <code>average_marks</code> and the drop fraction, using the function from Activity 3 (Sec. 3)</li>
<li>Build a pivot table with <code>region</code> as rows and <code>caste</code> as columns, mean <code>average_marks</code> in each cell (Sec. 4)</li>
</ol>
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>merged = pd.merge(df, schools, on="school_id", how="left")

regional_summary = merged.groupby("region").agg(
    mean_marks=("average_marks", "mean"),
    drop_fraction=("continue_drop", lambda s: (s == "drop").mean()),
)

region_caste_pivot = pd.pivot_table(merged, index="region", columns="caste", values="average_marks", aggfunc="mean")</pre>
</div>

In [ ]:
# TODO: build the regional performance report described above
...

## Summary

| Concept | Key rule |
|---|---|
| `pd.concat(axis=0)` | Stack rows; use when the same columns appear in separate batches |
| `pd.concat(axis=1)` | Stack columns by index position, not by key; prefer `merge` when there is a real key |
| `pd.merge(..., on=key, how=...)` | Combine two tables on a shared key, the same idea as a SQL join |
| `how="inner"` | Keep only rows with a match on both sides |
| `how="left"` | Keep every row from the left table; unmatched rows get `NaN` |
| `groupby(...)` | Returns a plan, not a result; nothing computes until you aggregate |
| `.agg([...])` after `groupby` | Several statistics per group, in one call |
| `pivot_table` | `groupby` with the second key laid out as columns instead of stacked as rows |

**Next:** `04-time-series.ipynb`, covering dates, `Timestamp`, `DatetimeIndex`, and `resample`.